> **Note**: This notebook requires raw data that is not included in this repository. To run it you need:
> - `data/raw/deployment_*/` — 2D annotation files (xlsx)
> - `data/raw/annotations_ipads.xlsx` — iPad corner annotations
> - `data/raw/response-times/` — manually annotated response frame files
> - `data/raw/video-positions/` — per-frame 3D centroid files
> - `data/stereo-parameters/` — stereo calibration files (.mat)
> 
> All other outputs are reproducible by running this notebook.

## Pipeline video processing

Overview - 1. Pipeline video processing

- Load packages
- Calculate 3D positions
	- Load stereo parameters
	- Calculate 3D positions of fish' head & tail
	- Calculate 3D positions of coral pieces
	- Calculate 3D position of iPad
- Calculate variables
	- Calculate Head Angle (HA) & Orientation Angle (OA)
	- Calculate Distance to Neighbours (DN) & Size (Sz)
	- Calculate Distance to Coral (DCo)
	- Calculate Distance to Centroid (DCe)
	- Calculate Perceived Loom Size (LSz) & Perceived Loom Speed (LSp)
	- Calculate Loom Side, NAS, RAS, N & RAShemi
	- Calculate Response Categories (ResC)
	- Calculate Time after First Responder (TFR)
	- Calculate Speed in 150ms after response (SpR)
- Construct summary per video
	- Calculate Group Size (GSz)
	- Calculate Relative Distance Coral & Loom (DCoCe)
	- Calculate Time to Collision (TCo)
	- Create checklist of fish with strange variables
	- Change fish categories to SRTC & NRTC if in checklist
- Update usable videos overview
    - Update based on summaries

### Load packages

In [ ]:
import numpy as np
import cv2 as cv
import scipy.io as sio
import pandas as pd
import os
from pathlib import Path
from math import acos, degrees
from scipy.spatial import distance
from scipy.spatial import Delaunay
from scipy.spatial.distance import euclidean
import math

In [3]:
sensory_motor_delay = 5  # minimum frames between first and secondary responder
non_stimulus_angle = 55
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
path = ROOT

### Calculate 3D positions

#### Load stereo parameters for the deployment

In [11]:
def load_parms(dep_id):

    # Load stereo parameters
    stereo_params = sio.loadmat(f'{path}/data/stereo-parameters/stereoparams_dep{dep_id}.mat')

    # Load stereo parameters
    K1 = stereo_params['intrinsicMatrix1']  # Camera 1 intrinsic matrix
    K2 = stereo_params['intrinsicMatrix2'] # Camera 2 intrinsic matrix
    R = stereo_params['rotationOfCamera2']   # Relative rotation matrix
    D1 = stereo_params['distortionCoefficients1'] # Distortion coefficients - camera 1
    D2 = stereo_params['distortionCoefficients2'] # Distortion coefficients - camera 2
    T = stereo_params['translationOfCamera2']   # Relative translation vector
    return(K1, K2, R, D1, D2, T)

#### Calculate 3D positions of fish (head or tail) - FRAMES

In [12]:
def calc_3D_fish(df, position, dep_id, video_id):
    
    K1, K2, R, D1, D2, T = load_parms(dep_id)
    
    df = df.reset_index(drop=True)

    # Create empty arrays for storing the 2D information
    pts1 = np.empty(shape=(df.shape[0],2))
    pts2 = np.empty(shape=(df.shape[0],2))

    # Load image points from both cameras
    for i, row in df.iterrows():
        pts1[i] = ([row['Cor_LF_' + str(position) + '_x'], row['Cor_LF_' + str(position) + '_y']])
        pts2[i] = ([row['Cor_RF_' + str(position) + '_x'], row['Cor_RF_' + str(position) + '_y']])

    # Normalize image points 
    pts1_norm = cv.undistortPoints(pts1, K1, D1, None, K1)
    pts2_norm = cv.undistortPoints(pts2, K2, D2, None, K2)

    T = T.reshape(3,1)

    # Create Projection matrices
    P1 = np.dot(K1, np.hstack((np.eye(3), np.zeros((3, 1)))))   # Projection matrix for camera 1
    P2 = np.dot(K2, np.hstack((R, T)))                          # Projection matrix for camera 2

    # Triangulate 3D points
    pts4D_hom = cv.triangulatePoints(P1, P2, pts1_norm, pts2_norm)
    pts3D_hom = pts4D_hom / pts4D_hom[3, :]
    points3D = pts3D_hom[:3, :].T

    points3D = np.hstack(((np.array([df['Fish_ID'], df['Species'], df['Video_ID']]).T), points3D))

    columns = ['Fish_ID', 'Species', 'Video_ID', 'X', 'Y', 'Z']
    points3D_df = pd.DataFrame(points3D, columns=columns)
    
    folder = f"{path}/data/derived/3D-positions-fish/Deployment{dep_id}/Video{video_id}"
    os.makedirs(folder, exist_ok=True)

    if position == 'Head':
        points3D_df.to_csv(f'{folder}/Heads_{video_id}.csv', index=False)
    elif position == 'Tail':
        points3D_df.to_csv(f'{folder}/Tails_{video_id}.csv', index=False)

    return points3D_df

#### Calculate 3D positions of coral (FRAMES, also works for VIDEO)

In [13]:
def calc_3D_surroundings(df, dep_id, coral_piece):

    K1, K2, R, D1, D2, T = load_parms(dep_id)

    df = df.reset_index(drop=True)

    # Create empty arrays for storing the 2D information
    pts1 = np.empty(shape=(df.shape[0],2))
    pts2 = np.empty(shape=(df.shape[0],2))

    # Load image points from both cameras
    for i, row in df.iterrows():
        pts1[i] = ([row['Cor_LF_x'], row['Cor_LF_y']])       # Set to names of camera 1
        pts2[i] = ([row['Cor_RF_x'], row['Cor_RF_y']])       # Set to names of camera 2

    # Normalize image points
    pts1_norm = cv.undistortPoints(pts1, K1, D1, None, K1)
    pts2_norm = cv.undistortPoints(pts2, K2, D2, None, K2)

    T = T.reshape(3,1)

    # Create Projection matrices
    P1 = np.dot(K1, np.hstack((np.eye(3), np.zeros((3, 1)))))   # Projection matrix for camera 1
    P2 = np.dot(K2, np.hstack((R, T)))                          # Projection matrix for camera 2

    # Triangulate 3D points
    pts4D_hom = cv.triangulatePoints(P1, P2, pts1_norm, pts2_norm)
    pts3D_hom = pts4D_hom / pts4D_hom[3, :]
    points3D = pts3D_hom[:3, :].T

    columns = ['X', 'Y', 'Z']
    points3D_df = pd.DataFrame(points3D, columns=columns)

    points3D_df.to_csv(f'{path}/data/derived/3D-positions-surroundings/Deployment{dep_id}/{coral_piece}_Dep{dep_id}.csv', index=False)

    return points3D_df

#### Calculate 3D position of ipad (FRAMES, also works for VIDEO)

In [14]:
def calc_3D_ipads_by_deployment(
    ipad_df: pd.DataFrame,
    path: str,
    save_corners_csv: bool = False,
    save_summary_csv: bool = False,
    save_legacy_xlsx: bool = True,
):
    """
    Compute iPad 3D info per deployment, supporting decimal-suffixed 'Deployment' like 10.1, 10.2.
    If an explicit 'Ipad_ID' column exists, it takes precedence; otherwise it is derived
    from the decimal part of 'Deployment'.

    Expected columns in ipad_df:
      - 'Deployment' (e.g. 3, "10.1", 10.2)
      - Optional 'Ipad_ID' (int)
      - For each corner C in {"Upper_left","Upper_right","Lower_left","Lower_right"}:
            LF_{C}_x, LF_{C}_y, RF_{C}_x, RF_{C}_y
      - Optional metadata: 'Video_nr', 'Object', etc. (these are carried through)

    Outputs per (DepBase, Ipad_ID):
      - (optional) {path}/DataFrames/3DPositionsIpad/Deployment{DepBase}/IpadCorners_Dep{DepBase}_Ipad{Ipad_ID}.csv
      - (optional) {path}/DataFrames/3DPositionsIpad/Deployment{DepBase}/IpadCornersSummary_Dep{DepBase}_Ipad{Ipad_ID}.csv
      - (legacy)   {path}/data/derived/3D-positions-surroundings/Deployment{DepBase}/ipad_{Ipad_ID}_coordinates_dep_{DepBase}.xlsx

    Returns:
      dict[(DepBase, Ipad_ID)] -> {
         "corners_df": per-row wide corners table,
         "calc_ipad_tall_df": tall per-row table (4 corners + centroid + vect_min),
         "summary_df": median corners + centroid + vect_min,
         "legacy_df": 6-row dataframe in legacy layout/order
      }
    """

    if "Deployment" not in ipad_df.columns:
        raise KeyError("ipad_df must contain a 'Deployment' column.")

    df = ipad_df.copy()

    # ---- Derive DepBase and Ipad_ID from 'Deployment' if needed ----
    def _parse_dep_ipad(val):
        s = str(val).strip()
        # Split on first dot, but keep robustness for ints/strings/floats
        if "." in s:
            left, right = s.split(".", 1)
        else:
            left, right = s, ""
        try:
            dep_base = int(float(left))
        except Exception as e:
            raise ValueError(f"Cannot parse DepBase from Deployment='{val}'") from e
        ipad_id = 1
        if right != "":
            try:
                ipad_id = int(float(right))
                if ipad_id <= 0:
                    ipad_id = 1
            except Exception:
                ipad_id = 1
        return dep_base, ipad_id

    if "Ipad_ID" not in df.columns:
        parsed = df["Deployment"].apply(_parse_dep_ipad)
        df["DepBase"] = [p[0] for p in parsed]
        df["Ipad_ID"] = [p[1] for p in parsed]
    else:
        # Respect provided Ipad_ID, derive DepBase from integer part of Deployment
        def _depbase(v):
            s = str(v).strip()
            if "." in s:
                s = s.split(".", 1)[0]
            return int(float(s))
        df["DepBase"] = df["Deployment"].apply(_depbase)

    # ---- Required corner columns ----
    corners = ["Upper_left", "Upper_right", "Lower_left", "Lower_right"]
    coord_cols = []
    for c in corners:
        coord_cols += [f"LF_{c}_x", f"LF_{c}_y", f"RF_{c}_x", f"RF_{c}_y"]

    # Coerce to numeric to avoid dtype('O') issues
    for col in coord_cols:
        if col not in df.columns:
            raise KeyError(f"Expected column '{col}' not found in ipad_df.")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    meta_cols = [c for c in ["Deployment", "DepBase", "Ipad_ID", "Video_nr", "Object"] if c in df.columns]

    results = {}

    # Legacy order & cross-product mapping to match your old files
    # p0=Lower_left, p1=Upper_right, p2=Lower_right
    legacy_order = ["Lower_left", "Upper_right", "Lower_right", "Upper_left"]

    for (dep_base, ipad_id), g in df.groupby(["DepBase", "Ipad_ID"], sort=True):
        # Drop rows with any missing coordinates
        g = g.dropna(subset=coord_cols).copy()
        if g.empty:
            continue

        # ---- Load stereo calibration for DepBase ----
        K1, K2, R, D1, D2, T = load_parms(dep_base)
        T = T.reshape(3, 1)

        # Projection matrices (undistortPoints with P=K)
        P1 = K1 @ np.hstack((np.eye(3), np.zeros((3, 1))))
        P2 = K2 @ np.hstack((R, T))

        # Triangulation helper
        def _triangulate_corner(corner_name):
            pts1 = np.column_stack([g[f"LF_{corner_name}_x"].to_numpy(),
                                    g[f"LF_{corner_name}_y"].to_numpy()]).astype(np.float32)
            pts2 = np.column_stack([g[f"RF_{corner_name}_x"].to_numpy(),
                                    g[f"RF_{corner_name}_y"].to_numpy()]).astype(np.float32)

            pts1_u = cv.undistortPoints(pts1.reshape(-1, 1, 2), K1, D1, P=K1).reshape(-1, 2)
            pts2_u = cv.undistortPoints(pts2.reshape(-1, 1, 2), K2, D2, P=K2).reshape(-1, 2)

            p1 = pts1_u.T  # (2, N)
            p2 = pts2_u.T  # (2, N)
            pts4D = cv.triangulatePoints(P1, P2, p1, p2)       # (4, N)
            pts3D = (pts4D[:3, :] / pts4D[3, :]).T             # (N, 3) float64
            return pts3D

        tri = {c: _triangulate_corner(c) for c in corners}

        # ---- Per-row wide corners table ----
        corners_df = g[meta_cols].reset_index(drop=True)
        for c in corners:
            corners_df[f"{c}_X"] = tri[c][:, 0]
            corners_df[f"{c}_Y"] = tri[c][:, 1]
            corners_df[f"{c}_Z"] = tri[c][:, 2]

        # ---- Tall per-row table: 4 corners + centroid + vect_min (legacy mapping) ----
        tall_rows = []
        for i in range(len(corners_df)):
            # Build a row-wise table in the legacy order so indices 0,1,2,3 match
            row_pts = []
            for name in legacy_order:
                row_pts.append({
                    "Point": name,
                    "X": float(corners_df.loc[i, f"{name}_X"]),
                    "Y": float(corners_df.loc[i, f"{name}_Y"]),
                    "Z": float(corners_df.loc[i, f"{name}_Z"]),
                })
            ipad_pts = pd.DataFrame(row_pts, columns=["Point", "X", "Y", "Z"])

            # Axis-aligned centroid from the 4 corners
            left   = float(ipad_pts["X"].min())
            right  = float(ipad_pts["X"].max())
            bottom = float(ipad_pts["Y"].min())
            top    = float(ipad_pts["Y"].max())
            front  = float(ipad_pts["Z"].min())
            back   = float(ipad_pts["Z"].max())

            x_center = left + (right - left) / 2.0
            y_center = bottom + (top - bottom) / 2.0
            z_center = front + (back - front) / 2.0

            ipad_pts.loc[len(ipad_pts)] = ["centroid", x_center, y_center, z_center]

            # Legacy cross-product: p0=Lower_left (row 0), p1=Upper_right (row 1), p2=Lower_right (row 2)
            p0 = ipad_pts.iloc[0, 1:4].to_numpy(dtype=float)
            p1 = ipad_pts.iloc[1, 1:4].to_numpy(dtype=float)
            p2 = ipad_pts.iloc[2, 1:4].to_numpy(dtype=float)
            up   = p2 - p0
            side = p1 - p0
            cross_ipad = -0.01 * np.cross(up, side)
            pointvect  = np.array([x_center, y_center, z_center], dtype=float) - cross_ipad
            ipad_pts.loc[len(ipad_pts)] = ["vect_min", pointvect[0], pointvect[1], pointvect[2]]

            # Attach metadata
            for m in meta_cols:
                ipad_pts[m] = corners_df.loc[i, m]
            ipad_pts["RowIndex"] = i
            tall_rows.append(ipad_pts)

        calc_ipad_tall_df = pd.concat(tall_rows, ignore_index=True)

        # ---- Summary via per-column median of corners ----
        summary_rows = []
        for name in legacy_order:
            summary_rows.append({
                "Point": name,
                "X": float(np.median(corners_df[f"{name}_X"])),
                "Y": float(np.median(corners_df[f"{name}_Y"])),
                "Z": float(np.median(corners_df[f"{name}_Z"])),
            })
        summary_df = pd.DataFrame(summary_rows, columns=["Point", "X", "Y", "Z"])

        # Add centroid and vect_min (same legacy mapping) to summary
        left   = float(summary_df["X"].min())
        right  = float(summary_df["X"].max())
        bottom = float(summary_df["Y"].min())
        top    = float(summary_df["Y"].max())
        front  = float(summary_df["Z"].min())
        back   = float(summary_df["Z"].max())

        x_center = left + (right - left) / 2.0
        y_center = bottom + (top - bottom) / 2.0
        z_center = front + (back - front) / 2.0
        summary_df.loc[len(summary_df)] = ["centroid", x_center, y_center, z_center]

        p0 = summary_df.iloc[0, 1:4].to_numpy(dtype=float)  # Lower_left
        p1 = summary_df.iloc[1, 1:4].to_numpy(dtype=float)  # Upper_right
        p2 = summary_df.iloc[2, 1:4].to_numpy(dtype=float)  # Lower_right
        up   = p2 - p0
        side = p1 - p0
        cross_ipad = -0.01 * np.cross(up, side)
        pointvect  = np.array([x_center, y_center, z_center], dtype=float) - cross_ipad
        summary_df.loc[len(summary_df)] = ["vect_min", pointvect[0], pointvect[1], pointvect[2]]

        summary_df["DepBase"] = dep_base
        summary_df["Ipad_ID"] = ipad_id

        # ---- Legacy 6-row file: rows 0..3, centroid, vect_min ----
        # Map the four corners to indices 0..3 in the *legacy order*,
        # then append centroid and vect_min.
        legacy_rows = []
        for i, name in enumerate(legacy_order):  # i = 0..3
            row = summary_df.loc[summary_df["Point"] == name].iloc[0]
            legacy_rows.append({"Point": i, "X": float(row["X"]), "Y": float(row["Y"]), "Z": float(row["Z"])})
        for name in ["centroid", "vect_min"]:
            row = summary_df.loc[summary_df["Point"] == name].iloc[0]
            legacy_rows.append({"Point": name, "X": float(row["X"]), "Y": float(row["Y"]), "Z": float(row["Z"])})
        legacy_df = pd.DataFrame(legacy_rows, columns=["Point", "X", "Y", "Z"])

        # ---- Save files ----
        out_dir_legacy = os.path.join(path, f"data/derived/3D-positions-surroundings/Deployment{dep_base}")
        os.makedirs(out_dir_legacy, exist_ok=True)

        if save_corners_csv:
            corners_path = os.path.join(out_dir_legacy, f"IpadCorners_Dep{dep_base}_Ipad{ipad_id}.csv")
            corners_df.to_csv(corners_path, index=False)

        if save_summary_csv:
            summary_path = os.path.join(out_dir_legacy, f"IpadCornersSummary_Dep{dep_base}_Ipad{ipad_id}.csv")
            summary_df.to_csv(summary_path, index=False)

        if save_legacy_xlsx:
            legacy_xlsx = os.path.join(out_dir_legacy, f"iPad{ipad_id}_coordinates_Dep{dep_base}.csv")
            # Use Excel (legacy expectation); switch to .csv if you prefer
            legacy_df.to_csv(legacy_xlsx, index=False)

        # ---- Collect results ----
        results[(dep_base, ipad_id)] = {
            "corners_df": corners_df,
            "calc_ipad_tall_df": calc_ipad_tall_df,
            "summary_df": summary_df,
            "legacy_df": legacy_df,
        }

    return results


### Calculate distances, sizes, angles and variables

#### Calculate distance to neighbours (DN) and size (Sz) - FRAMES

In [15]:
def calc_distances(heads_df, tails_df, dep_id, video_id):
    columns = ['Dep_ID', 'Video_ID', 'Fish_ID'] + ['Size (in mm)'] + ['Distance_to_' + str(i) for i in heads_df['Fish_ID']]
    distances_df = pd.DataFrame(columns=columns)
    distances_df['Dep_ID'] = [dep_id for x in range(len(heads_df))]
    distances_df['Video_ID'] = [video_id for x in range(len(heads_df))]
    distances_df['Fish_ID'] = heads_df['Fish_ID']

    for index, row in heads_df.iterrows():
        heads_xyz = heads_df.loc[index, ['X', 'Y', 'Z']]
        tails_xyz = tails_df.loc[index, ['X', 'Y', 'Z']]
        if np.isnan(heads_xyz['X']):
            distances_df.at[index, 'Size (in mm)'] = np.nan
        elif np.isnan(tails_xyz['X']):
            distances_df.at[index, 'Size (in mm)'] = np.nan
            for index1, row1 in heads_df.iterrows():
                heads_neighbour_xyz = heads_df.loc[index1, ['X', 'Y', 'Z']]
                tails_neighbour_xyz = tails_df.loc[index1, ['X', 'Y', 'Z']]
                if np.isnan(heads_neighbour_xyz['X']) and np.isnan(tails_neighbour_xyz['X']):
                    distances_df.at[index, 'Distance_to_' + str(index1 + 1)] = np.nan
                elif np.isnan(heads_neighbour_xyz['X']):
                    distances_df.at[index, 'Distance_to_' + str(index1 + 1)] = distance.euclidean(tails_neighbour_xyz, heads_xyz)
                elif np.isnan(tails_neighbour_xyz['X']):
                    distances_df.at[index, 'Distance_to_' + str(index1 + 1)] = distance.euclidean(heads_neighbour_xyz, heads_xyz)
                else:
                    distances_df.at[index, 'Distance_to_' + str(index1 + 1)] = min(distance.euclidean(heads_neighbour_xyz, heads_xyz), distance.euclidean(tails_neighbour_xyz, heads_xyz))                
        else:
            distances_df.at[index, 'Size (in mm)'] = distance.euclidean(tails_df.loc[index, ['X', 'Y', 'Z']], heads_df.loc[index, ['X', 'Y', 'Z']])
            for index1, row1 in heads_df.iterrows():
                heads_neighbour_xyz = heads_df.loc[index1, ['X', 'Y', 'Z']]
                tails_neighbour_xyz = tails_df.loc[index1, ['X', 'Y', 'Z']]
                if np.isnan(heads_neighbour_xyz['X']) and np.isnan(tails_neighbour_xyz['X']):
                    distances_df.at[index, 'Distance_to_' + str(index1 + 1)] = np.nan
                elif np.isnan(heads_neighbour_xyz['X']):
                    distances_df.at[index, 'Distance_to_' + str(index1 + 1)] = distance.euclidean(tails_neighbour_xyz, heads_xyz)
                elif np.isnan(tails_neighbour_xyz['X']):
                    distances_df.at[index, 'Distance_to_' + str(index1 + 1)] = distance.euclidean(heads_neighbour_xyz, heads_xyz)
                else:
                    distances_df.at[index, 'Distance_to_' + str(index1 + 1)] = min(distance.euclidean(heads_neighbour_xyz, heads_xyz), distance.euclidean(tails_neighbour_xyz, heads_xyz))

    distances_df.to_csv(f'{path}/data/derived/3D-positions-fish/Deployment{dep_id}/Video{video_id}/Distances_{video_id}.csv', index=False)

    return distances_df

#### Calculate viewing angle (VA) and orientation angle (OA) of fish - FRAMES

In [16]:
def calc_angle(heads_df, tails_df, dep_id, video_id, ipad_id):
    if ipad_id == 1:
        ipad_df = pd.read_csv(f'{path}/data/derived/3D-positions-surroundings/Deployment{dep_id}/iPad1_coordinates_Dep{dep_id}.csv')
    elif ipad_id == 2:
        ipad_df = pd.read_csv(f'{path}/data/derived/3D-positions-surroundings/Deployment{dep_id}/iPad2_coordinates_Dep{dep_id}.csv')

    centroid = np.array(ipad_df.loc[ipad_df['Point'] == 'centroid', ['X', 'Y', 'Z']])[0]
    pointvect = np.array(ipad_df.loc[ipad_df['Point'] == 'vect_min', ['X', 'Y', 'Z']])[0]

    line_ipad = pointvect - centroid
    mag_vec_ipad = np.sqrt(np.sum(line_ipad**2))

    # create a dataframe with all head_angles and orientation_angles of the fish
    columns = ['Fish_ID', 'Head_Angle', 'Orient_Angle']
    angles_df = pd.DataFrame(columns=columns)
    orient_angle = []
    head_angle = []

    for index, row in heads_df.iterrows():
        head_xyz = np.array(row[['X', 'Y', 'Z']])
        tail_xyz = np.array(tails_df.loc[index, ['X', 'Y', 'Z']])

        # Calculate vectors
        vector_fish = head_xyz - tail_xyz
        vector_head_center = head_xyz - centroid
        vector_ipad = line_ipad

        # Normalize vectors
        vector_fish_normalized = vector_fish / np.linalg.norm(vector_fish)
        vector_ipad_normalized = vector_ipad / np.linalg.norm(vector_ipad)

        # Calculate the dot product
        cos_theta = np.dot(vector_fish_normalized, vector_ipad_normalized)

        # Calculate the angle in radians, and then convert to degrees
        angle_i = np.degrees(np.arccos(cos_theta))

        # Append the orientation angle to the list
        orient_angle.append(angle_i)

        # Calculate dot product between vector_head_center and vector_ipad
        dot_head_i = np.sum(vector_head_center * line_ipad)

        # Append the head angle to the list
        head_angle.append(degrees(acos(dot_head_i / (np.sqrt(np.sum(vector_head_center**2)) * mag_vec_ipad))))

    angles_df['Fish_ID'] = heads_df['Fish_ID']
    angles_df['Head_Angle'] = head_angle
    angles_df['Orient_Angle'] = orient_angle


    angles_df.to_csv(f'{path}/data/derived/3D-positions-fish/Deployment{dep_id}/Video{video_id}/Angles_{video_id}.csv', index=False)

    return angles_df

#### Calculate all needed 3D information - FRAMES

In [17]:
def calculate_3D_information(annotations_df, surroundings_df, ipad_df, dep_id, video_id, ipad_id, calculate_3D_again):
    if calculate_3D_again == 'Yes':
        heads_df = calc_3D_fish(annotations_df, 'Head', dep_id, video_id)
        tails_df = calc_3D_fish(annotations_df, 'Tail', dep_id, video_id)
    else:
        heads_df = pd.read_csv(f'{path}/data/derived/3D-positions-fish/Deployment{dep_id}/Video{video_id}/Heads_{video_id}.csv')
        tails_df = pd.read_csv(f'{path}/data/derived/3D-positions-fish/Deployment{dep_id}/Video{video_id}/Tails_{video_id}.csv')

    # include 'Coral' once, then Coral_1..Coral_4
    for type_label in ['Coral'] + [f'Coral_{i}' for i in range(1, 5)]:
        mask = surroundings_df['Type'].astype(str).eq(type_label)
        if mask.any():
            coral_df = surroundings_df.loc[mask]
            # Pass the full label to your function so filenames are correct:
            coral_3D_coordinates = calc_3D_surroundings(coral_df, dep_id, coral_piece=type_label)

    ipad_3D_coordinates = calc_3D_ipads_by_deployment(ipad_df, path=path, save_corners_csv=False, save_summary_csv=False, save_legacy_xlsx=True)

    distances_df = calc_distances(heads_df, tails_df, dep_id, video_id)
    angles_df = calc_angle(heads_df, tails_df, dep_id, video_id, ipad_id)

    return heads_df, tails_df, ipad_3D_coordinates, coral_3D_coordinates, distances_df, angles_df

In [ ]:
def calculate_dataframes(usable_videos):
    for index, row in usable_videos.iterrows():
        annotations_df = pd.read_excel(f'{path}/data/raw/deployment_{row['Dep_ID']}/annotations_{row['Video_ID']}.xlsx')
        surroundings_df = pd.read_excel(f'{path}/data/raw/deployment_{row["Dep_ID"]}/Surroundings_{row["Dep_ID"]}.xlsx')
        ipad_df = pd.read_excel(f'{path}/data/raw/annotations_ipads.xlsx')

        calculate_3D_information(annotations_df, surroundings_df, ipad_df, row['Dep_ID'], row['Video_ID'], row['Ipad_ID'], 'Yes')  

In [19]:
usable_videos = pd.read_excel(f'{path}/data/usable_videos.xlsx')

In [20]:
calculate_dataframes(usable_videos)

Video: 1
Video: 2
Video: 3
Video: 4
Video: 5
Video: 6
Video: 7
Video: 9
Video: 11
Video: 12
Video: 14
Video: 15
Video: 16
Video: 17
Video: 18
Video: 19
Video: 20
Video: 23
Video: 26
Video: 28
Video: 29
Video: 30
Video: 31
Video: 33
Video: 34
Video: 40
Video: 41
Video: 42
Video: 43
Video: 44
Video: 45
Video: 48
Video: 50
Video: 52
Video: 53
Video: 54
Video: 55
Video: 57
Video: 58
Video: 59
Video: 60
Video: 61
Video: 62
Video: 63
Video: 65
Video: 66
Video: 69
Video: 71
Video: 73
Video: 76
Video: 77
Video: 78
Video: 79
Video: 80
Video: 81
Video: 82
Video: 84
Video: 85
Video: 86
Video: 87
Video: 88
Video: 89
Video: 90
Video: 91
Video: 92
Video: 93
Video: 94
Video: 95
Video: 102
Video: 103
Video: 112
Video: 113
Video: 114
Video: 115
Video: 116
Video: 117
Video: 118
Video: 119
Video: 120
Video: 121
Video: 122
Video: 123
Video: 124
Video: 125
Video: 126
Video: 127
Video: 128
Video: 130
Video: 132
Video: 133
Video: 134
Video: 135
Video: 136
Video: 137
Video: 139
Video: 140
Video: 141
Video: 14

#### Calculate distance to coral (DCo) - FRAMES

In [21]:
def load_coral_files(folder_path):
    coral_data = {}
    for file in os.listdir(folder_path):
        if file.startswith("Coral_") and file.endswith(".csv"):
            file_path = os.path.join(folder_path, file)
            try:
                coral_id = file.split('.')[0]  # or any other way to identify the coral
                coral_data[coral_id] = pd.read_csv(file_path)
            except Exception as e:
                print(f"Could not read file {file_path}: {e}")
                continue  # Skip to the next file
    return coral_data

In [22]:
def same_side(p1, p2, a, b):
    p1 = pd.to_numeric(p1)
    p2 = pd.to_numeric(p2)
    a = pd.to_numeric(a)
    b = pd.to_numeric(b)

    cp1 = np.cross(b - a, p1 - a)
    cp2 = np.cross(b - a, p2 - a)
    
    return np.dot(cp1, cp2) >= 0

In [23]:
def nearest_point_on_edge(point, vertices):
    min_distance = float('inf')
    nearest_point = None

    for i in range(3):
        p1, p2 = vertices[i], vertices[(i+1)%3]
        edge_vec = p2 - p1
        point_vec = point - p1
        t = np.dot(point_vec, edge_vec) / np.dot(edge_vec, edge_vec)
        t = max(0, min(1, t))  # Clamp t to the range [0, 1]
        closest = p1 + t * edge_vec

        distance = euclidean(point, closest)
        if distance < min_distance:
            min_distance = distance
            nearest_point = closest

    return nearest_point

In [24]:
def point_to_triangle_distance(point, triangle):
    # Unpack the triangle vertices
    p1, p2, p3 = triangle

    # Compute the normal of the plane
    normal = np.cross(p2 - p1, p3 - p1)
    normal /= np.linalg.norm(normal)

    # Compute the projection of the point onto the plane
    proj = point - np.dot(point - p1, normal) * normal

    # Check if the projection lies inside the triangle
    # This involves checking if the point is on the same side of each edge
    
    if (same_side(proj, p1, p2, p3) and
        same_side(proj, p2, p1, p3) and
        same_side(proj, p3, p1, p2)):
        # If inside the triangle, return distance to the plane
        return euclidean(point, proj), proj
    
    else:
        # Find the nearest point on each edge of the triangle
        nearest_on_edge = nearest_point_on_edge(point, [p1, p2, p3])
        return euclidean(point, nearest_on_edge), nearest_on_edge

In [25]:
def unique_rows(data):
    uniq, index = np.unique(data.reshape(-1, np.prod(data.shape[1:])), axis=0, return_index=True)
    return uniq[index.argsort()].reshape(-1, data.shape[1], data.shape[2])

In [26]:
def triangulate_corals(coral_data):
    coral_triangles = {}
    for coral_id, data in coral_data.items():
        points = data[['X', 'Y', 'Z']].to_numpy()
        tri = Delaunay(points)
        tetras = points[tri.simplices]

        faces = np.vstack([tetras[:, [0, 1, 2]], tetras[:, [0, 1, 3]], tetras[:, [0, 2, 3]], tetras[:, [1, 2, 3]]])

        unique_faces = unique_rows(faces)
        coral_triangles[coral_id] = unique_faces

    return coral_triangles

In [27]:
def find_closest_coral_point(fish_head, coral_triangles):
    min_distance = float('inf')
    closest_coral_id = None
    closest_point = None

    for coral_id, triangles in coral_triangles.items():
        coral_id = coral_id.split('_')[1]
        if 'Dep' in coral_id:
            coral_id = 1
        for triangle in triangles:
            distance, triangle_closest_point = point_to_triangle_distance(fish_head.to_numpy(), triangle)
            if distance < min_distance:
                min_distance = distance
                closest_coral_id = coral_id
                closest_point = triangle_closest_point
    
    return closest_coral_id, min_distance, closest_point

In [28]:
def calculate_coral_information(heads_df, coral_dict):
    coral_triangles = triangulate_corals(coral_dict)

    closest_corals = []
    distance_coral = []
    x_list = []
    y_list = []
    z_list = []

    for index, row in heads_df.iterrows():
        try:
            closest_corals.append(find_closest_coral_point(row[['X', 'Y', 'Z']], coral_triangles)[0])
            distance_coral.append(find_closest_coral_point(row[['X', 'Y', 'Z']], coral_triangles)[1])
            x_list.append(find_closest_coral_point(row[['X', 'Y', 'Z']], coral_triangles)[2][0])
            y_list.append(find_closest_coral_point(row[['X', 'Y', 'Z']], coral_triangles)[2][1])
            z_list.append(find_closest_coral_point(row[['X', 'Y', 'Z']], coral_triangles)[2][2])
        except:
            closest_corals.append(np.nan)
            distance_coral.append(np.nan)
            x_list.append(np.nan)
            y_list.append(np.nan)
            z_list.append(np.nan)
    
    return closest_corals, distance_coral, x_list, y_list, z_list
        

#### Calculate distance to stimulus centroid (DSt) - FRAMES

In [29]:
def calculate_3d_distance(point1, heads_df):
    distance_list = []
    
    x1, y1, z1 = point1

    for index, row in heads_df.iterrows():
        x2, y2, z2 = row[['X', 'Y', 'Z']]
        distance = math.sqrt((x2 - x1)**2 + (y2 - y1)**2 + (z2 - z1)**2)
        distance_list.append(distance)

    return distance_list

#### Calculate response categories (ResC)

In [30]:
def calculate_response_categories(summary_df):
    categories = []
    minimal_responsetime = summary_df['Responseframe_LF'].min() + sensory_motor_delay
    
    for index, row in summary_df.iterrows():
        if row['Responseframe_LF'] > 0:
            if row['Responseframe_LF'] < minimal_responsetime:
                categories.append('FR')
            elif row['Responseframe_LF'] >= minimal_responsetime and row['Viewing_Angle'] > non_stimulus_angle:
                categories.append('SRNL')
            elif row['Responseframe_LF'] >= minimal_responsetime and row['Viewing_Angle'] < non_stimulus_angle:
                categories.append('SRL')
            else:
                categories.append('SRUD')
        else:
            if (row['Viewing_Angle'] > non_stimulus_angle) & (row['Response_Video'] == 'Yes'):
                categories.append('NRNL_r')
            elif (row['Viewing_Angle'] > non_stimulus_angle) & (row['Response_Video'] == 'No'):
                categories.append('NRNL_nr')
            elif (row['Viewing_Angle'] < non_stimulus_angle) & (row['Response_Video'] == 'Yes'):
                categories.append('NRL_r')
            elif (row['Viewing_Angle'] < non_stimulus_angle) & (row['Response_Video'] == 'No'):
                categories.append('NRL_nr')
            else:
                categories.append('NRUD')
    
    return(categories)

#### Calculate time after FR (TFR)

In [31]:
def after_FR(summary_df):
    after_FR_times = []
    for index, row in summary_df.iterrows():
        if row['Response_Present'] == 'Yes':
            after_FR_times.append(row['Responseframe_LF'] - summary_df['Responseframe_LF'].min(skipna=True))
        elif row['Response_Present'] == 'No' or row['Response_Present'] == 'Maybe':
            after_FR_times.append(summary_df['Responseframe_LF'].max(skipna=True) - summary_df['Responseframe_LF'].min(skipna=True) + 1)
        
    return after_FR_times

#### Calculate speed in 150ms after response (SpR) - VIDEOS

In [ ]:
def calculate_speeds(summary_df, dep_id, video_id):
    speeds = []
    time_seconds = 0.15
    for index, row in summary_df.iterrows():
        if row['Response_Category'] in ['FR', 'SRL', 'SRNL']:
            response_frame = int(20 + row['After_FR']) # FR is at frame 20
            after_response_150 = int(response_frame + (time_seconds * 240))
            position_before_response_df = pd.read_excel(f'{path}/data/raw/video-positions/deployment_{dep_id}/video_{video_id}/centroids_frame_{response_frame}.xlsx')
            position_after_response_df = pd.read_excel(f'{path}/data/raw/video-positions/deployment_{dep_id}/video_{video_id}/centroids_frame_{after_response_150}.xlsx')
            position_before = position_before_response_df.loc[position_before_response_df['Fish_ID'] == row['Fish_ID'], ['X', 'Y', 'Z']]
            position_after = position_after_response_df.loc[position_after_response_df['Fish_ID'] == row['Fish_ID'], ['X', 'Y', 'Z']]
            if position_before.empty or position_after.empty:
                print(f'No track for video {video_id}, fish {row["Fish_ID"]}')
                speeds.append(np.nan)
            elif not position_before.empty and not position_after.empty:
                distance_mm = euclidean(position_before.values[0], position_after.values[0])
                speeds.append(distance_mm/time_seconds)
        else:
            speeds.append(np.nan)
    return speeds

#### Calculate Neighbour Proximity - FRAMES

In [33]:
def calculate_local_density(distances_df, col_pattern='Distance_to', distance_unit='mm'):
    # pick all neighbor-distance columns
    cols = distances_df.columns[distances_df.columns.str.contains(col_pattern)]
    D = distances_df[cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)

    # treat zeros as missing to avoid 1/0
    D[D == 0] = np.nan

    # convert to meters if your distances are in mm
    if distance_unit == 'mm':
        D_m = D / 1000.0
    elif distance_unit == 'm':
        D_m = D
    else:
        raise ValueError("distance_unit must be 'mm' or 'm'")

    # LocalDensity_i = sum_j (1 / d_ij^2) over neighbors (ignoring NaNs)
    local_density = np.nansum((1.0 / D_m) ** 2, axis=1)

    # return a Series aligned to the original rows
    return pd.Series(local_density, index=distances_df.index, name='LocalDensity')

### Construct summary for every video - FRAMES

In [34]:
def construct_summaries(usable_videos):
    for index, row in usable_videos.iterrows():
        print('Video: ', row['Video_ID'])

        columns = ['Dep_ID', 'Video_ID', 'Response_Video', 'Group_Size', 'Loom_Speed', 'Site', 'Site_Rep', 'Trial_Event',
                   'Fish_ID', 'Species', 'Size', 'Response_Present', 'Responseframe_LF', 'Responseframe_cam3', 'After_FR',
                   'Response_Rank', 'Response_Category', 'Speed', 'Closest_Coral', 'Distance_to_Coral', 'X_Coral', 'Y_Coral', 'Z_Coral',
                   'Distance_to_Stimulus', 'Neighbour_Proximity', 'Orientation_Angle', 'Viewing_Angle']

        summary_df = pd.DataFrame(columns=columns)

        annotations_df = pd.read_excel(f'{path}/data/raw/deployment_{row['Dep_ID']}/annotations_{row['Video_ID']}.xlsx')
        distances_df = pd.read_csv(f'{path}/data/derived/3D-positions-fish/Deployment{row['Dep_ID']}/Video{row['Video_ID']}/Distances_{row['Video_ID']}.csv')
        angles_df = pd.read_csv(f'{path}/data/derived/3D-positions-fish/Deployment{row['Dep_ID']}/Video{row['Video_ID']}/Angles_{row['Video_ID']}.csv')
        heads_df = pd.read_csv(f'{path}/data/derived/3D-positions-fish/Deployment{row['Dep_ID']}/Video{row['Video_ID']}/Heads_{row["Video_ID"]}.csv')
        tails_df = pd.read_csv(f'{path}/data/derived/3D-positions-fish/Deployment{row['Dep_ID']}/Video{row['Video_ID']}/Tails_{row["Video_ID"]}.csv')
        if row['Video_ID'] in list(np.arange(333, 364)):
            ipad_coordinates = pd.read_csv(f'{path}/data/derived/3D-positions-surroundings/Deployment{row['Dep_ID']}/iPad2_coordinates_Dep{row['Dep_ID']}.csv')
        else:
            ipad_coordinates = pd.read_csv(f'{path}/data/derived/3D-positions-surroundings/Deployment{row['Dep_ID']}/iPad1_coordinates_Dep{row['Dep_ID']}.csv')
        coral_dict = load_coral_files(f'{path}/data/derived/3D-positions-surroundings/Deployment{row['Dep_ID']}')

        offsets_videos = pd.read_excel(f'{path}/data/raw/offsets_videos.xlsx')

        if row['Response'] == 'Yes':
            response_df = pd.read_excel(f'{path}/data/raw/response-times/deployment_{row['Dep_ID']}/responsetimes_{row['Video_ID']}.xlsx')

        summary_df['Dep_ID'] = annotations_df['Dep_ID'].copy()
        summary_df['Video_ID'] = annotations_df['Video_ID'].copy()
        summary_df['Fish_ID'] = annotations_df['Fish_ID'].copy()
        summary_df['Group_Size'] = len(summary_df)
        summary_df['Loom_Speed'] = row['Loom_Speed']
        summary_df['Site'] = row['Site']
        summary_df['Site_Rep'] = row['Site_Rep']
        summary_df['Trial_Event'] = row['Dep_Event']
        summary_df['Species'] = annotations_df['Species'].copy()
        summary_df['Size'] = distances_df['Size (in mm)'].copy()

        if row['Response'] == 'No':
            summary_df['Response_Present'] = ['No' for i in range(len(annotations_df))]
            summary_df['Responseframe_LF'] = [np.nan for i in range(len(annotations_df))]
            summary_df['Responseframe_cam3'] = [np.nan for i in range(len(annotations_df))]
            summary_df['After_FR'] = [np.nan for i in range(len(annotations_df))]
            summary_df['Response_Rank'] = [np.nan for i in range(len(annotations_df))]

        elif row['Response'] == 'Yes':
            summary_df['Response_Present'] = response_df['Response_Present'].copy()
            summary_df['Responseframe_LF'] = response_df['Responseframe_LF'].copy()
            summary_df['Responseframe_cam3'] = response_df['Responseframe_LF'].copy() - offsets_videos.loc[offsets_videos['Video_ID'] == row['Video_ID'], 'Offset_1_3'].values[0]

            summary_df['Response_Rank'] = np.where(summary_df['Responseframe_LF'] < summary_df['Responseframe_LF'].min() + sensory_motor_delay, 1, np.nan)
            summary_df['Response_Rank'] = summary_df['Response_Rank'].fillna(summary_df['Responseframe_LF'].rank(method='dense', na_option='top'))
            summary_df['Response_Rank'] = np.where(summary_df['Response_Present'] == 'No', summary_df['Response_Rank'].max() + 1, summary_df['Response_Rank'])
            summary_df['Response_Rank'] = summary_df['Response_Rank'].rank(method='dense').astype(int)
        
        summary_df['Response_Video'] = 'Yes' if (summary_df['Response_Present'] == 'Yes').any() else 'No'
        summary_df['After_FR'] = after_FR(summary_df)
        summary_df['Orientation_Angle'] = angles_df['Orient_Angle'].copy()
        summary_df['Viewing_Angle'] = angles_df['Head_Angle'].copy()
        summary_df['Response_Category'] = calculate_response_categories(summary_df)
        summary_df['Speed'] = calculate_speeds(summary_df, row['Dep_ID'], row['Video_ID'])
        
        closest_coral, distance_coral, x_coral, y_coral, z_coral = calculate_coral_information(heads_df, coral_dict)

        summary_df['Closest_Coral'] = closest_coral
        summary_df['Distance_to_Coral'] = distance_coral
        summary_df['X_Coral'] = x_coral
        summary_df['Y_Coral'] = y_coral
        summary_df['Z_Coral'] = z_coral
        summary_df['Distance_to_Stimulus'] = calculate_3d_distance(ipad_coordinates.loc[ipad_coordinates['Point'] == 'centroid', ['X', 'Y', 'Z']].values[0], heads_df)
        summary_df['Neighbour_Proximity'] = calculate_local_density(distances_df)

        summary_df['Loom_Speed'] = usable_videos.loc[usable_videos['Video_ID'] == row['Video_ID'], 'Loom_Speed'].values[0]

        folder = f'{path}/data/derived/summaries/Deployment{row["Dep_ID"]}'
        if not os.path.exists(folder):
            os.makedirs(folder, exist_ok=True)
        summary_df.to_csv(f'{path}/data/derived/summaries/Deployment{row["Dep_ID"]}/Summary_Video{row["Video_ID"]}.csv')

#### Make list to check

In [35]:
def construct_listtocheck(usable_videos):
    dep_ids = []
    video_ids = []
    fish_ids = []
    reponse_categories = []
    sizes = []
    speeds = []

    for index, row in usable_videos.iterrows():
        summary_df = pd.read_csv(f'{path}/data/derived/summaries/Deployment{row["Dep_ID"]}/Summary_Video{row["Video_ID"]}.csv')

        for index_sum, row_sum in summary_df.iterrows():
            if row_sum['Size'] > 500:
                dep_ids.append(row_sum['Dep_ID'])
                video_ids.append(row_sum['Video_ID'])
                fish_ids.append(row_sum['Fish_ID'])
                reponse_categories.append(row_sum['Response_Category'])
                sizes.append(row_sum['Size'])
                speeds.append(row_sum['Speed'])
            if row_sum['Response_Category'] in ['FR', 'SRL', 'SRNL']:
                if math.isnan(row_sum['Speed']) or row_sum['Speed'] > 5000:
                    dep_ids.append(row_sum['Dep_ID'])
                    video_ids.append(row_sum['Video_ID'])
                    fish_ids.append(row_sum['Fish_ID'])
                    reponse_categories.append(row_sum['Response_Category'])
                    sizes.append(row_sum['Size'])
                    speeds.append(row_sum['Speed'])
            elif (math.isnan(row_sum['Size'])) & (row_sum['Response_Category'] not in ['NRUD', 'SRUD']):
                dep_ids.append(row_sum['Dep_ID'])
                video_ids.append(row_sum['Video_ID'])
                fish_ids.append(row_sum['Fish_ID'])
                reponse_categories.append(row_sum['Response_Category'])
                sizes.append(row_sum['Size'])
                speeds.append(row_sum['Speed'])

    df = pd.DataFrame({'Dep_ID': dep_ids, 'Video_ID': video_ids, 'Fish_ID': fish_ids, 'Response_Category': reponse_categories, 'Size': sizes, 'Speed': speeds})
    df.to_csv(f'{path}/data/derived/to_check.csv', index=False)
    return df

#### Change fish categories to SRTC and NRTC if in checklist

In [36]:
def change_categories(list_to_check):
    for index, row in list_to_check.iterrows():
        summary_df = pd.read_csv(f'{path}/data/derived/summaries/Deployment{row["Dep_ID"]}/Summary_Video{row["Video_ID"]}.csv')
        for index_sum, row_sum in summary_df.iterrows():
            if row_sum['Fish_ID'] == row['Fish_ID']:
                if row_sum['Response_Category'] in ['NRL_r', 'NRNL_r', 'NRL_nr', 'NRNL_nr']:
                    summary_df.at[index_sum, 'Response_Category'] = 'NRTC'
                elif row_sum['Response_Category'] in ['SRL', 'SRNL']:
                    summary_df.at[index_sum, 'Response_Category'] = 'SRTC'
                elif row_sum['Response_Category'] in ['FR']:
                    summary_df.at[index_sum, 'Response_Category'] = 'FRTC'

        summary_df.to_csv(f'{path}/data/derived/summaries/Deployment{row["Dep_ID"]}/Summary_Video{row["Video_ID"]}.csv', index=False)

### Run all calculations

In [37]:
usable_videos = pd.read_excel(f'{path}/data/usable_videos.xlsx')
response_videos = usable_videos.loc[usable_videos['Response'] == 'Yes']

# Change categories for list to check
list_to_check = construct_listtocheck(usable_videos)
change_categories(list_to_check)

### Create overall Summary-file for all fish

In [38]:
def make_overall_summary(usable_videos):
    summaries_list = []
    for index, row in usable_videos.iterrows():
        summary = pd.read_csv(f'{path}/data/derived/summaries/Deployment{row['Dep_ID']}/Summary_Video{row['Video_ID']}.csv')
        summaries_list.append(summary)
    overall_summaries = pd.concat(summaries_list, ignore_index=True)
    overall_summaries['Response_Binary'] = np.where(overall_summaries['Response_Category'].isin(['FR', 'SRL', 'SRNL', 'FRTC', 'SRTC']), 1, 0)

    overall_summaries.to_csv(f'{path}/data/all_observations.csv', index=False)

### Create overall Distance-file for all fish

In [39]:
def make_overall_distances(usable_videos):
    distances_list = []
    for index, row in usable_videos.iterrows():
        distances = pd.read_csv(f'{path}/data/derived/3D-positions-fish/Deployment{row['Dep_ID']}/Video{row['Video_ID']}/Distances_{row['Video_ID']}.csv')
        distances_list.append(distances)
    overall_distances = pd.concat(distances_list, ignore_index=True)

    overall_distances.to_csv(f'{path}/data/derived/all_distances.csv', index=False)

In [ ]:
def make_overall_positions(usable_videos):
    """Concatenate per-video Heads_*.csv files into a single all_positions.csv."""
    positions_list = []
    for index, row in usable_videos.iterrows():
        heads = pd.read_csv(f'{path}/data/derived/3D-positions-fish/Deployment{row["Dep_ID"]}/Video{row["Video_ID"]}/Heads_{row["Video_ID"]}.csv')
        positions_list.append(heads)
    all_positions = pd.concat(positions_list, ignore_index=True)
    all_positions.to_csv(f'{path}/data/derived/all_positions.csv', index=False)

In [40]:
make_overall_summary(usable_videos)
make_overall_distances(usable_videos)
make_overall_positions(usable_videos)


## Create 3D positions tracking files for videos

In [12]:
# Merge all files of a video into one

def merge_files(dep_id, video_id):
    video_path = path / f'data/raw/video-positions/deployment_{dep_id}/video_{video_id}'
    all_files = [f for f in os.listdir(video_path) if f.startswith('centroids_frame_') and f.endswith('.xlsx')]
    all_files.sort(key=lambda x: int(x.split('_')[2].split('.')[0]))  # Sort by frame number

    merged_df = pd.DataFrame()
    for file in all_files:
        df = pd.read_excel(os.path.join(video_path, file))
        frame_number = int(file.split('_')[2].split('.')[0])
        df['Frame_ID'] = frame_number
        merged_df = pd.concat([merged_df, df], ignore_index=True)

    merged_df.to_csv(video_path / f'Centroids_Video_Overall_{video_id}.csv', index=False)

In [ ]:
for index, row in response_videos.iterrows():
    merge_files(row['Dep_ID'], row['Video_ID'])

Merging files for Video: 14
Merging files for Video: 15
Merging files for Video: 16
Merging files for Video: 20
Merging files for Video: 23
Merging files for Video: 26
Merging files for Video: 29
Merging files for Video: 33
Merging files for Video: 41
Merging files for Video: 42
Merging files for Video: 43
Merging files for Video: 48
Merging files for Video: 54
Merging files for Video: 58
Merging files for Video: 60
Merging files for Video: 61
Merging files for Video: 62
Merging files for Video: 92
Merging files for Video: 116
Merging files for Video: 122
Merging files for Video: 134
Merging files for Video: 143
Merging files for Video: 168
Merging files for Video: 175
Merging files for Video: 183
Merging files for Video: 186
Merging files for Video: 187
Merging files for Video: 189
Merging files for Video: 191
Merging files for Video: 194
Merging files for Video: 195
Merging files for Video: 198
Merging files for Video: 202
Merging files for Video: 204
Merging files for Video: 206
Mer